# Generate fact_transactions
Creates a 20,000-row fact table and writes data/fact_transactions.csv.
Foreign keys are sampled only from keys present in existing dimension CSVs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
FACT_ROWS = 20000
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'README.md').exists() and (candidate / '01_data').exists():
            return candidate
    return current

def clipped_normal(rng: np.random.Generator, mean: float, std_dev: float, size: int, min_value: float, max_value: float) -> np.ndarray:
    values = rng.normal(loc=mean, scale=std_dev, size=size)
    return np.clip(values, min_value, max_value)

project_root = find_project_root()
data_dir = project_root / '01_data' / 'raw'

dim_date = pd.read_csv(data_dir / 'dim_date.csv')
dim_customer = pd.read_csv(data_dir / 'dim_customer.csv')
dim_account = pd.read_csv(data_dir / 'dim_account.csv')
dim_branch = pd.read_csv(data_dir / 'dim_branch.csv')
dim_product = pd.read_csv(data_dir / 'dim_product.csv')
dim_transaction_type = pd.read_csv(data_dir / 'dim_transaction_type.csv')
dim_currency = pd.read_csv(data_dir / 'dim_currency.csv')

amount = clipped_normal(rng, mean=250.0, std_dev=120.0, size=FACT_ROWS, min_value=1.0, max_value=20000.0)
fee = clipped_normal(rng, mean=3.5, std_dev=1.2, size=FACT_ROWS, min_value=0.01, max_value=250.0)
credit_score = clipped_normal(rng, mean=680.0, std_dev=90.0, size=FACT_ROWS, min_value=300.0, max_value=850.0)

fact_transactions = pd.DataFrame({
    'transaction_id': np.arange(1, FACT_ROWS + 1, dtype=np.int64),
    'date_key': rng.choice(dim_date['date_key'].to_numpy(), size=FACT_ROWS, replace=True),
    'customer_key': rng.choice(dim_customer['customer_key'].to_numpy(), size=FACT_ROWS, replace=True),
    'account_key': rng.choice(dim_account['account_key'].to_numpy(), size=FACT_ROWS, replace=True),
    'branch_key': rng.choice(dim_branch['branch_key'].to_numpy(), size=FACT_ROWS, replace=True),
    'product_key': rng.choice(dim_product['product_key'].to_numpy(), size=FACT_ROWS, replace=True),
    'transaction_type_key': rng.choice(dim_transaction_type['transaction_type_key'].to_numpy(), size=FACT_ROWS, replace=True),
    'currency_key': rng.choice(dim_currency['currency_key'].to_numpy(), size=FACT_ROWS, replace=True),
    'amount': np.round(amount, 2),
    'fee': np.round(fee, 2),
    'credit_score': np.round(credit_score).astype(np.int32),
})

data_dir.mkdir(parents=True, exist_ok=True)
output_path = data_dir / 'fact_transactions.csv'
fact_transactions.to_csv(output_path, index=False)

print(f'Rows: {len(fact_transactions)}')
print(f'Wrote: {output_path}')
fact_transactions.head()